# 01 GPU Env Setup

학습용 `Python 3.10` conda env를 만들고, Jupyter 커널 등록과 GPU 0 고정까지 확인하는 노트북이다.

설명 문서:

- `../README.md`
- `../docs/GUIDE.md`
- `../docs/gpu-env-setup.md`

## 1. 현재 커널 확인

In [ ]:
import os, sys, socket
print('hostname =', socket.gethostname())
print('python =', sys.executable)
print('pid =', os.getpid())
print('cwd =', os.getcwd())

## 2. conda 확인

In [ ]:
!which conda
!conda --version
!python --version

## 3. 서버 전체 GPU 상태 확인

여기서는 서버 전체 GPU가 보인다.

In [ ]:
!nvidia-smi

## 4. Python 3.10 conda env 생성

In [ ]:
!conda create -n ems-ai-py310 python=3.10 -y

## 5. env Python 버전 확인

In [ ]:
!conda run -n ems-ai-py310 python --version

## 6. ipykernel 설치 및 커널 등록

In [ ]:
!conda run -n ems-ai-py310 pip install ipykernel
!conda run -n ems-ai-py310 python -m ipykernel install --user --name ems-ai-py310 --display-name "Python (ems-ai-py310)"

## 7. 일반 학습 패키지 설치

In [ ]:
!conda run -n ems-ai-py310 pip install -r ../requirements-train.txt

## 8. PyTorch 설치

In [ ]:
!conda run -n ems-ai-py310 pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

## 9. 새 커널로 변경

이 시점부터 Jupyter 커널을 `Python (ems-ai-py310)`로 변경한 뒤 아래 셀을 실행한다.

## 10. 새 커널 첫 셀: GPU 0 고정

반드시 `torch` import 전에 실행한다.

In [4]:
import os
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print('CUDA_DEVICE_ORDER =', os.environ.get('CUDA_DEVICE_ORDER'))
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES'))

CUDA_DEVICE_ORDER = PCI_BUS_ID
CUDA_VISIBLE_DEVICES = 0


## 11. torch와 CUDA 확인

In [9]:
import torch, os, sys
print('python =', sys.executable)
print('pid =', os.getpid())
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
print('device count =', torch.cuda.device_count())
if torch.cuda.is_available():
    print('current device =', torch.cuda.current_device())
    print('device name =', torch.cuda.get_device_name(0))

python = /home/j-k14s305/.conda/envs/ems-ai-py310/bin/python
pid = 2239590
torch = 2.6.0+cu124
cuda available = True
device count = 1
current device = 0
device name = NVIDIA H200 NVL


## 12. 실제 GPU 0 사용 확인

In [11]:
import torch, os, time
x = torch.randn(4096, 4096, device='cuda')
print('pid =', os.getpid())
print('tensor device =', x.device)
time.sleep(30)

pid = 2239590
tensor device = cuda:0


In [12]:
!nvidia-smi
import sys
!{sys.executable} -c "import torch; print(torch.__version__, torch.cuda.is_available())"


Mon Apr 27 10:14:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200 NVL                On  |   00000000:15:00.0 Off |                    0 |
| N/A   33C    P0             93W /  450W |     591MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [13]:
import os, sys
from pathlib import Path

repo = Path("/home/j-k14s305/S14P31S305")  # 실제 repo 위치로 맞춰야 함
os.chdir(repo)

os.environ["PYTHONPATH"] = str(repo / "ems/ai")
os.environ["S305_AI_DATA_ROOT"] = "/data/s305-ai-data"      # 서버에 데이터 둔 경로로 수정
os.environ["S305_AI_OUTPUT_ROOT"] = "/data/s305-ai-runs"    # 결과 저장 경로

print(os.getcwd())
print(sys.executable)


FileNotFoundError: [Errno 2] No such file or directory: '/home/j-k14s305/S14P31S305'

In [54]:
!mkdir -p /home/j-k14s305/S14P31S305/ems
!unzip -o /home/j-k14s305/ems-ai.zip -d /home/j-k14s305/S14P31S305/ems

!mkdir -p /home/j-k14s305/s305-ai-data
!unzip -o /home/j-k14s305/s305-ai-data-processed.zip -d /home/j-k14s305/s305-ai-data

/bin/bash: line 1: unzip: command not found
/bin/bash: line 1: unzip: command not found


In [55]:
from pathlib import Path

print(Path("/home/j-k14s305/S14P31S305/ems/ai/train/train.py").exists())
print(Path("/home/j-k14s305/S14P31S305/ems/ai/configs/solar_kpx_baseline_gpu.yaml").exists())
print(Path("/home/j-k14s305/s305-ai-data/processed/splits/solar_kpx_train.csv").exists())
print(Path("/home/j-k14s305/s305-ai-data/processed/splits/solar_kpx_val.csv").exists())


False
False
False
False


In [68]:
from pathlib import Path
import zipfile

home = Path("/home/j-k14s305")
work = home / "s305-work"
repo_parent = work / "repo-fixed"
data_root = work / "data-fixed"

repo_parent.mkdir(parents=True, exist_ok=True)
data_root.mkdir(parents=True, exist_ok=True)

def extract_zip_normalized(zip_path: Path, target: Path):
    with zipfile.ZipFile(zip_path, "r") as z:
        for info in z.infolist():
            name = info.filename.replace("\\", "/")
            if not name or name.endswith("/"):
                continue
            out = target / name
            out.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, out.open("wb") as dst:
                dst.write(src.read())

extract_zip_normalized(home / "S14P31S305.zip", repo_parent)
extract_zip_normalized(home / "s305-ai-data-processed.zip", data_root)


In [69]:
from pathlib import Path

work = Path("/home/j-k14s305/s305-work")

for p in work.rglob("train.py"):
    if "ems" in str(p):
        print("train.py =", p)

for p in work.rglob("solar_kpx_baseline_gpu.yaml"):
    print("config =", p)

for p in work.rglob("solar_kpx_train.csv"):
    print("train csv =", p)

for p in work.rglob("solar_kpx_val.csv"):
    print("val csv =", p)


train.py = /home/j-k14s305/s305-work/repo-fixed/S14P31S305/ems/ai/train/train.py
config = /home/j-k14s305/s305-work/repo-fixed/S14P31S305/ems/ai/configs/solar_kpx_baseline_gpu.yaml
train csv = /home/j-k14s305/s305-work/data-fixed/processed/splits/solar_kpx_train.csv
val csv = /home/j-k14s305/s305-work/data-fixed/processed/splits/solar_kpx_val.csv


In [70]:
import os, sys
from pathlib import Path

repo = Path("/home/j-k14s305/s305-work/repo-fixed/S14P31S305")
data_root = Path("/home/j-k14s305/s305-work/data-fixed")
run_root = Path("/home/j-k14s305/s305-work/runs")

os.chdir(repo)
sys.path.insert(0, str(repo / "ems/ai"))

os.environ["PYTHONPATH"] = str(repo / "ems/ai")
os.environ["S305_AI_DATA_ROOT"] = str(data_root)
os.environ["S305_AI_OUTPUT_ROOT"] = str(run_root)

print("repo =", repo)
print("data =", data_root)
print("runs =", run_root)

repo = /home/j-k14s305/s305-work/repo-fixed/S14P31S305
data = /home/j-k14s305/s305-work/data-fixed
runs = /home/j-k14s305/s305-work/runs


In [71]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.6.0+cu124
True
NVIDIA H200 NVL


In [72]:
!{sys.executable} -m train.train --config ems/ai/configs/solar_kpx_baseline_gpu.yaml

2026-04-27 10:42:25,689 | INFO | epoch=1 train_loss=174194134721.153076 val_loss=37906929527.466667 val_rmse=194697.025432
2026-04-27 10:42:25,828 | INFO | epoch=2 train_loss=36005529959.688629 val_loss=17443280600.177776 val_rmse=132073.005751
2026-04-27 10:42:25,964 | INFO | epoch=3 train_loss=27809127672.289368 val_loss=15584268834.133333 val_rmse=124836.966833
2026-04-27 10:42:26,099 | INFO | epoch=4 train_loss=25452022783.295834 val_loss=14510963666.488890 val_rmse=120461.461522
2026-04-27 10:42:26,234 | INFO | epoch=5 train_loss=23390229193.603081 val_loss=14211067744.711111 val_rmse=119210.183726
2026-04-27 10:42:26,369 | INFO | epoch=6 train_loss=23021756459.095036 val_loss=13450607274.666666 val_rmse=115976.754636
2026-04-27 10:42:26,523 | INFO | epoch=7 train_loss=21632804772.739925 val_loss=13575844886.755556 val_rmse=116515.427579
2026-04-27 10:42:26,665 | INFO | epoch=8 train_loss=21578482879.110989 val_loss=12728655689.955555 val_rmse=112821.349469
2026-04-27 10:42:26,800

In [73]:
from pathlib import Path

run_root = Path("/home/j-k14s305/s305-work/runs")
print(list((run_root / "checkpoints/solar_kpx_baseline").glob("*")))
print((run_root / "logs/solar_kpx_baseline/train.log").read_text()[-2000:])

[PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_001.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_004.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_008.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_009.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/latest.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_011.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_019.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_006.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_010.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_020.pt'), PosixPath('/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/epoch_007.pt'), PosixPath('/home/j-k14s

In [74]:
import torch
from pathlib import Path

ckpt = torch.load(
    "/home/j-k14s305/s305-work/runs/checkpoints/solar_kpx_baseline/best.pt",
    map_location="cpu",
)
print(ckpt["epoch"])
print(ckpt["metrics"])

19
{'epoch': 19, 'train_loss': 19817898690.98391, 'train_mae': 88561.640625, 'train_rmse': 140776.06341988684, 'train_mape': 128069465600.0, 'train_masked_mape_target_gte_1': 19559.500122070312, 'val_loss': 11196756673.422222, 'val_mae': 67777.4375, 'val_rmse': 105814.72956068073, 'val_mape': 107829632000.0, 'val_masked_mape_target_gte_1': 8828.244018554688}


In [94]:
import os
import subprocess
from pathlib import Path

WORKDIR = Path("/home/j-k14s305/s305-work")

os.chdir(WORKDIR)
Path("runs").mkdir(exist_ok=True)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["S305_AI_DATA_ROOT"] = str(WORKDIR / "s305-ai-data")
os.environ["S305_AI_OUTPUT_ROOT"] = str(WORKDIR / "runs")
os.environ["PYTHONPATH"] = str(WORKDIR / "ems" / "ai")

cmd = ["bash", "ems/ai/scripts/run_training_stages.sh"]

process = subprocess.Popen(
    cmd,
    cwd=WORKDIR,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nreturn_code =", return_code)

if return_code != 0:
    raise RuntimeError(f"Training failed with return_code={return_code}")


[stage-1] MLP solar KPX baseline
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/j-k14s305/s305-work/ems/ai/train/train.py", line 6, in <module>
    import numpy as np
ModuleNotFoundError: No module named 'numpy'

return_code = 1


RuntimeError: Training failed with return_code=1

In [102]:
import os
import sys
import subprocess
from pathlib import Path

WORKDIR = Path("/home/j-k14s305/s305-work")

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["S305_AI_DATA_ROOT"] = str(WORKDIR / "s305-ai-data")
env["S305_AI_OUTPUT_ROOT"] = str(WORKDIR / "runs")
env["PYTHONPATH"] = str(WORKDIR / "ems" / "ai")

cmd = [
    sys.executable,
    "-m",
    "train.lightgbm_train",
    "--config",
    "ems/ai/configs/solar_kpx_lightgbm_gpu.yaml",
]

process = subprocess.Popen(
    cmd,
    cwd=WORKDIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("return_code =", return_code)

if return_code != 0:
    raise RuntimeError(f"LightGBM failed with return_code={return_code}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.058133 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2949
[LightGBM] [Info] Number of data points in the train set: 7271, number of used features: 15
[LightGBM] [Info] Start training from score 498926.214270
Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 130653	valid_0's l2: 1.70702e+10
[100]	valid_0's rmse: 81818.1	valid_0's l2: 6.69421e+09
[150]	valid_0's rmse: 80837.9	valid_0's l2: 6.53476e+09
Early stopping, best iteration is:
[116]	valid_0's rmse: 80302.5	valid_0's l2: 6.44849e+09
{
  "run_name": "solar_kpx_lightgbm",
  "model_path": "/home/j-k14s305/s305-work/runs/artifacts/solar_kpx_lightgbm/model.joblib",
  "train_rows_before_dropna": 7271,
  "train_rows_after_dropna": 7271,
  "val_rows_before_dropna": 1440,
  "val_rows_after_dropna": 1440,
  "metrics": {
    "mae": 52539.15397232863,
    "rmse": 80302.48315

In [104]:
import json
from pathlib import Path

metrics_path = Path("/home/j-k14s305/s305-work/runs/artifacts/solar_kpx_lightgbm/metrics.json")

with metrics_path.open("r", encoding="utf-8") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))


{
  "run_name": "solar_kpx_lightgbm",
  "model_path": "/home/j-k14s305/s305-work/runs/artifacts/solar_kpx_lightgbm/model.joblib",
  "train_rows_before_dropna": 7271,
  "train_rows_after_dropna": 7271,
  "val_rows_before_dropna": 1440,
  "val_rows_after_dropna": 1440,
  "metrics": {
    "mae": 52539.15397232863,
    "rmse": 80302.48315849618,
    "mape": 1.1088270480460452e+21,
    "masked_mape_target_gte_1": 17327.278265092868,
    "clipped_mae": 52539.15397232863,
    "clipped_rmse": 80302.48315849618,
    "clipped_masked_mape_target_gte_1": 17327.278265092868,
    "postprocessed_mae": 52539.15397232863,
    "postprocessed_rmse": 80302.48315849618,
    "postprocessed_masked_mape_target_gte_1": 17327.278265092868
  }
}
